In [ ]:
# Cargar los paquetes requeridos
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import fetch_openml

In [ ]:
diabetes = fetch_openml(name='diabetes', version=1, as_frame=True)
df = diabetes.frame
df

In [ ]:
df.info()

In [ ]:
# Separar las características (X) y la variable objetivo (y)
# Obtener X eliminando la columna de etiqueta
X = df.drop(columns=['class'])

# Obtener Y convirtiendo la etiqueta en un valor numérico
y = df['class'].apply(lambda x: 1 if x == 'tested_positive' else 0)



In [ ]:
# Dividir el dataset en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=13,stratify=y)


In [ ]:
# Estandarizar las características (importante para el perceptrón)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)
X_train

In [ ]:
# Crear y entrenar el perceptrón
perceptron = Perceptron(max_iter=4000, eta0=0.01, random_state=13)
perceptron.fit(X_train, y_train)

In [ ]:
y_pred = perceptron.predict(X_test)

In [ ]:
eficiencia = accuracy_score(y_test, y_pred)
print(f'Precisión del perceptrón: {eficiencia:.2f}')
print('\nReporte de clasificación: ')
print(classification_report(y_test, y_pred))

In [ ]:
# Obtener los pesos (matriz de coeficientes)
pesos = perceptron.coef_
# Obtener el sesgo (término de sesgo o bias)
sesgo = perceptron.intercept_


In [ ]:
# Mostrar los resultados
print(f"Matriz de pesos: {pesos}")
print(f'\nVector de sesgos: {sesgo}')

## Gráfica de la frontera de decisión

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Seleccionar solo dos características [glucosa e IMC]
X_subset = X_train[:,[1,5]]
y_subset = y_train

In [ ]:
perceptron.fit(X_subset, y_subset)

In [ ]:
# Obtener pesos y bias
w = perceptron.coef_
b = perceptron.intercept_
w

In [ ]:
# Crear una línea de decisión
x1_min, x1_max = X_subset[:,0].min(), X_subset[:,0].max()
x2_min, x2_max = X_subset[:,1].min(), X_subset[:,1].max()
x1_values = np.linspace(x1_min, x1_max, 100)
x2_values = (-w[0,0]/w[0,1]) * x1_values - (b[0]/w[0,1])
x2_values


In [ ]:
# Graficar puntos de datos
plt.figure(figsize=(8,6))
plt.scatter(X_subset[y_subset==0][:,0], X_subset[y_subset==0][:,1], color='blue', label='No diabético')

plt.scatter(X_subset[y_subset==1][:,0], X_subset[y_subset==1][:,1], color='red', label='Diabético')
plt.ylim(-5,5)

plt.plot(x1_values, x2_values, 'k-', label='Frontera de decisión');

In [ ]:
from sklearn.neural_network import MLPClassifier

# Crear un perceptrón multicapa con 2 capa oculta de 5 neuronas
mlp = MLPClassifier(hidden_layer_sizes=(5,), activation='logistic', max_iter=1500, random_state=15)

# Entrenar el modelo
mlp.fit(X_train, y_train)

# Verificar la cantidad de capas
print(f"Número de capas en la red: {len(mlp.coefs_)}. (Incluye capas ocultas y capa de salida)")
print(f'Neuronas en la capa oculta: {mlp.hidden_layer_sizes[0]}')


In [ ]:
# Obtener pesos y bias
w = mlp.coefs_
b = mlp.intercepts_
print(f'Pesos:\n{w}')
print(f'Interceptos\n{b}')

In [ ]:
# Realizar predicciones
y_pred2 = mlp.predict(X_test)


In [ ]:
eficiencia2 = accuracy_score(y_test, y_pred2)
print(f'Eficiencia del MLP: {eficiencia2:.2f}')
print('\nReporte de clasificación:')
print(classification_report(y_test, y_pred2))

### Aplicación de CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv1D, MaxPool1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

### Reorganización del conjunto a #D

In [ ]:
X_train = X_train.reshape(614,8,1)
X_test = X_test.reshape(154,8,1)
X_train.shape, X_test.shape

In [ ]:
# Preparación del modelo
model = Sequential()


In [ ]:
model.add(Conv1D(filters=16, kernel_size=2, activation='relu', input_shape=(8,1)))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Conv1D(8, 2, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Flatten())
model.add(Dense(8, 'relu'))
model.add(Dropout(0.2))

model.add(Dense(1, 'sigmoid'))

In [ ]:
# Resumen
model.summary()

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy',metrics=['accuracy'])